In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve, roc_curve, auc
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import cycle

In [ ]:
# Load dataset
df = pd.read_csv("/kaggle/input/medicinal-plants-datasetwith1000rec/medicinal_plants_datasetlow.csv")

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
df['Crop'] = label_encoder.fit_transform(df['Crop'])

In [ ]:
# Split features and target
X = df.drop(columns=['Crop'])
y = df['Crop']

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Split into train (60%), validation (20%), and test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

In [ ]:
# Define model creation function
def create_model(input_shape, num_classes):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_shape,)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_temp, y_temp), 1):
    print(f'Fold {fold}')
    X_train_fold = X_temp[train_idx]
    y_train_fold = y_temp.iloc[train_idx]
    X_val_fold = X_temp[val_idx]
    y_val_fold = y_temp.iloc[val_idx]

    # Train model
    model = create_model(X_train.shape[1], len(label_encoder.classes_))
    history = model.fit(X_train_fold, y_train_fold, epochs=20, batch_size=64,
                        validation_data=(X_val_fold, y_val_fold), verbose=0)

    # Evaluate
    _, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f'Fold {fold} Validation Accuracy: {val_accuracy * 100:.2f}%')
    cv_scores.append(val_accuracy)

print(f'Average 5-Fold CV Accuracy: {np.mean(cv_scores) * 100:.2f}% (± {np.std(cv_scores) * 100:.2f}%)')

In [ ]:
# Train final model on full train+validation set
final_model = create_model(X_train.shape[1], len(label_encoder.classes_))
history = final_model.fit(X_temp, y_temp, epochs=20, batch_size=64,
                          validation_data=(X_val, y_val), verbose=0)

In [ ]:
# Plot Training and Validation Accuracy
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.savefig('accuracy_plot.png')
plt.close()

In [ ]:
# Plot Training and Validation Loss
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('loss_plot.png')
plt.close()

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = final_model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

In [ ]:
# Predictions
y_pred = final_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
plt.close()

In [ ]:
# Calculate TP, TN, FP, FN per class
tp = np.diag(cm)  # True Positives (diagonal)
fp = cm.sum(axis=0) - tp  # False Positives (column sum - TP)
fn = cm.sum(axis=1) - tp  # False Negatives (row sum - TP)
tn = cm.sum() - (tp + fp + fn)  # True Negatives (total - TP - FP - FN)

# Print TP, TN, FP, FN for each class
print("\nTP, TN, FP, FN per Class:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name}:")
    print(f"  True Positives (TP): {tp[i]}")
    print(f"  True Negatives (TN): {tn[i]}")
    print(f"  False Positives (FP): {fp[i]}")
    print(f"  False Negatives (FN): {fn[i]}")

# Bar Plot for TP, TN, FP, FN
metrics = ['TP', 'TN', 'FP', 'FN']
n_classes = len(label_encoder.classes_)
x = np.arange(n_classes)
width = 0.2

plt.figure(figsize=(12, 8))
plt.bar(x - 1.5*width, tp, width, label='True Positives', color='green')
plt.bar(x - 0.5*width, tn, width, label='True Negatives', color='blue')
plt.bar(x + 0.5*width, fp, width, label='False Positives', color='red')
plt.bar(x + 1.5*width, fn, width, label='False Negatives', color='orange')
plt.xlabel('Classes')
plt.ylabel('Count')
plt.title('True Positives, True Negatives, False Positives, and False Negatives per Class')
plt.xticks(x, label_encoder.classes_, rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('tp_tn_fp_fn_plot.png')
plt.close()

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

In [ ]:
# Precision-Recall Curve per Class
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown'])
y_test_bin = tf.keras.utils.to_categorical(y_test, num_classes=len(label_encoder.classes_))

for i, color in zip(range(len(label_encoder.classes_)), colors):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_pred[:, i])
    plt.plot(recall, precision, color=color, lw=2,
             label=f'{label_encoder.classes_[i]} (area = {np.trapz(precision, recall):.2f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve per Class')
plt.legend(loc='best')
plt.savefig('precision_recall_curve.png')
plt.close()

In [ ]:
# ROC Curve per Class
plt.figure(figsize=(10, 8))
for i, color in zip(range(len(label_encoder.classes_)), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{label_encoder.classes_[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve per Class')
plt.legend(loc='best')
plt.savefig('roc_curve.png')
plt.close()

In [ ]:
# Save model
final_model.save('deep_learning_medicinal_plants_model.h5')

In [ ]:
# Example prediction
static_input = np.array([[40.68, 39.89, 65.45, 30.46, 62.00, 342.59]])
static_input_scaled = scaler.transform(static_input)
prediction = final_model.predict(static_input_scaled)
predicted_class = np.argmax(prediction, axis=1)
predicted_crop = label_encoder.inverse_transform(predicted_class)
print(f'Predicted Crop: {predicted_crop[0]}')